# Notebook 1: SV2A Membrane Protein System Preparation for MD Simulation

**Protein:** Synaptic Vesicle Glycoprotein 2A (SV2A)  
**PDB ID:** 8UO9 (Cryo-EM structure)  
**Force Field:** AMBER14 + Lipid17 (via OpenMM)  
**Membrane:** POPC bilayer  
**Platform:** CPU (preparation) → GPU (production)  

---

## Prerequisites
- Conda environment: `mddock`
- Required packages: `openmm`, `requests`, `numpy`
- OS: Windows 10/11

Run this notebook inside your `mddock` environment:
```
conda activate mddock
jupyter notebook
```

---
## Cell 1: Install Required Packages (Run Once)

In [ ]:
# Run this cell ONCE to install missing packages in your mddock environment
# If openmm is already installed, this will just confirm it

import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

try:
    import openmm
    print(f"OpenMM version: {openmm.__version__} ✓")
except ImportError:
    print("OpenMM not found. Install via conda:")
    print("  conda install -c conda-forge openmm")

try:
    import requests
    print(f"requests ✓")
except ImportError:
    install("requests")
    print("requests installed ✓")

try:
    import numpy
    print(f"numpy version: {numpy.__version__} ✓")
except ImportError:
    install("numpy")
    print("numpy installed ✓")

---
## Cell 2: Set Working Directory

In [ ]:
import os

# Set your working directory here
work_dir = r"/workspace/SV2A_MD"

# Create directory if it doesn't exist
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

print(f"Working directory: {os.getcwd()}")

---
## Cell 3: Download SV2A Structure from RCSB PDB

We download PDB **8UO9** (SV2A cryo-EM structure). Since OPM may require manual download, we'll get it from RCSB and handle orientation in the next step.

In [ ]:
import requests

pdb_id = "8UO9"
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"

print(f"Downloading {pdb_id} from RCSB PDB...")
response = requests.get(url)

if response.status_code == 200:
    with open(f"{pdb_id}.pdb", "w") as f:
        f.write(response.text)
    print(f"✓ Downloaded: {pdb_id}.pdb ({len(response.text)} bytes)")
else:
    print(f"✗ Failed to download. Status code: {response.status_code}")
    print("  Please download manually from: https://www.rcsb.org/structure/8UO9")

---
## Cell 4: Clean the PDB File

We extract only the **SV2A protein chain (Chain A)**, removing:
- Nanobody/antibody chains
- Ligands and heteroatoms (optional: keep if you want ligand-bound simulation)
- Water molecules from crystallography
- DUM atoms (if downloaded from OPM)

In [ ]:
# Cell 4: Clean PDB and center protein at origin along Z-axis
import numpy as np

input_pdb = "8UO9.pdb"
output_pdb = "sv2a_clean.pdb"

# ========================================
# CONFIGURE
# ========================================
target_chain = "A"
keep_ligands = False
# ========================================

atom_lines = []
coords = []

with open(input_pdb, "r") as f:
    for line in f:
        if line.startswith("ATOM"):
            chain_id = line[21].strip()
            if chain_id == target_chain and "DUM" not in line:
                atom_lines.append(line)
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                coords.append([x, y, z])

coords = np.array(coords)
center = coords.mean(axis=0)
print(f"Original center of mass: X={center[0]:.2f}, Y={center[1]:.2f}, Z={center[2]:.2f}")

# Center the protein at origin (0, 0, 0)
# This is CRITICAL for addMembrane() to work
shifted_coords = coords - center
new_center = shifted_coords.mean(axis=0)
print(f"New center of mass: X={new_center[0]:.2f}, Y={new_center[1]:.2f}, Z={new_center[2]:.2f}")

# Write centered PDB
with open(output_pdb, "w") as f:
    f.write("CRYST1  150.000  150.000  150.000  90.00  90.00  90.00 P 1           1\n")
    f.write("MODEL        1\n")
    for i, line in enumerate(atom_lines):
        x, y, z = shifted_coords[i]
        # Replace coordinates in PDB line (columns 31-54)
        new_line = line[:30] + f"{x:8.3f}{y:8.3f}{z:8.3f}" + line[54:]
        f.write(new_line)
    f.write("TER\n")
    f.write("ENDMDL\n")
    f.write("END\n")

print(f"\n✓ Cleaned and centered PDB saved: {output_pdb}")
print(f"  Chain: {target_chain}")
print(f"  Atoms: {len(atom_lines)}")
print(f"  Protein is now centered at origin (0, 0, 0)")


In [ ]:
# ========================================
# CONFIGURE: Which chain is SV2A?
# ========================================
target_chain = "A"  # Change this if SV2A is on a different chain
keep_ligands = False  # Set True if you want to keep bound ligands (e.g., levetiracetam)

# ========================================

atom_lines = []
removed_het = 0

with open(input_pdb, "r") as f:
    for line in f:
        if line.startswith("ATOM"):
            chain_id = line[21].strip()
            if chain_id == target_chain:
                # Skip DUM atoms (from OPM files)
                if "DUM" not in line:
                    atom_lines.append(line)
        elif line.startswith("HETATM") and keep_ligands:
            chain_id = line[21].strip()
            resname = line[17:20].strip()
            if chain_id == target_chain and resname != "HOH" and "DUM" not in line:
                atom_lines.append(line)
        elif line.startswith("HETATM"):
            removed_het += 1

# Write properly formatted PDB
with open(output_pdb, "w") as f:
    f.write("CRYST1  120.000  120.000  120.000  90.00  90.00  90.00 P 1           1\n")
    f.write("MODEL        1\n")
    for line in atom_lines:
        f.write(line)
    f.write("TER\n")
    f.write("ENDMDL\n")
    f.write("END\n")

print(f"✓ Cleaned PDB saved: {output_pdb}")
print(f"  Chain kept: {target_chain}")
print(f"  Protein atoms: {len(atom_lines)}")
print(f"  HETATM removed: {removed_het}")

---
## Cell 5: Load Structure into OpenMM and Verify

This verifies that OpenMM can correctly parse the cleaned PDB.

In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *

print("Loading sv2a_clean.pdb into OpenMM...")
pdb = PDBFile('sv2a_clean.pdb')

print(f"✓ Successfully loaded!")
print(f"  Number of atoms: {pdb.topology.getNumAtoms()}")
print(f"  Number of residues: {pdb.topology.getNumResidues()}")
print(f"  Number of chains: {pdb.topology.getNumChains()}")

# Print first and last residues
residues = list(pdb.topology.residues())
print(f"  First residue: {residues[0].name} {residues[0].id}")
print(f"  Last residue: {residues[-1].name} {residues[-1].id}")

---
## Cell 6: Fix Missing Atoms and Add Hydrogens

Cryo-EM structures often have missing residues/atoms. OpenMM's `Modeller` can add missing hydrogens and fix common issues. If there are missing loops, we handle them here.

In [ ]:
import sys
!{sys.executable} -m pip install pdbfixer

In [ ]:
from pdbfixer import PDBFixer

print("PDBFixer installed successfully")

In [ ]:
# Cell 6: Fix missing atoms/terminals and add hydrogens
from pdbfixer import PDBFixer
from openmm.app import *
from openmm.unit import *

print("Step 1: Fixing missing atoms and terminals with PDBFixer...")
fixer = PDBFixer(filename='sv2a_clean.pdb')

# MUST call findMissingResidues first (even if you don't add them)
fixer.findMissingResidues()
print(f"  Missing residues (loops): {len(fixer.missingResidues)} gaps found")

# Remove missing residues from the dict so they are NOT modeled
# (modeling long missing loops is unreliable — skip them)
fixer.missingResidues = {}

# Now find missing heavy atoms (requires missingResidues to exist)
fixer.findMissingAtoms()
print(f"  Missing atoms: {len(fixer.missingAtoms)} residues affected")

# Add missing heavy atoms (terminal oxygens, etc.)
fixer.addMissingAtoms()
print("  ✓ Missing atoms added")

# Add hydrogens at pH 7.0
fixer.addMissingHydrogens(7.0)
print("  ✓ Hydrogens added at pH 7.0")

# Save the fixed structure
with open('sv2a_protonated.pdb', 'w') as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)

print(f"\n✓ Fixed PDB saved: sv2a_protonated.pdb")
print(f"  Total atoms: {fixer.topology.getNumAtoms()}")
print(f"  Total residues: {fixer.topology.getNumResidues()}")


In [ ]:
# Cell 6b: Verify protein orientation and check Z-extent
from openmm.app import *
from openmm.unit import *
import numpy as np

pdb = PDBFile('sv2a_protonated.pdb')

# Get all positions as numpy array
positions = np.array([[p.x, p.y, p.z] for p in pdb.positions.value_in_unit(nanometers)])

z_min = positions[:, 2].min()
z_max = positions[:, 2].max()
z_center = positions[:, 2].mean()
z_extent = z_max - z_min

x_extent = positions[:, 0].max() - positions[:, 0].min()
y_extent = positions[:, 1].max() - positions[:, 1].min()

print("Protein dimensions:")
print(f"  X extent: {x_extent:.2f} nm")
print(f"  Y extent: {y_extent:.2f} nm")
print(f"  Z extent: {z_extent:.2f} nm")
print(f"  Z center: {z_center:.2f} nm")
print(f"  Z range:  {z_min:.2f} to {z_max:.2f} nm")

print(f"\n  A typical membrane is ~4 nm thick.")
print(f"  The protein Z-center should be near 0.0 nm.")

if abs(z_center) > 1.0:
    print(f"\n  ⚠️ WARNING: Z-center is far from 0! This may cause addMembrane() to fail.")
    print(f"  The protein needs to be re-centered.")
else:
    print(f"\n  ✓ Protein is well-centered for membrane embedding.")


---
## Cell 7: Build POPC Membrane + Solvate + Add Ions

**This is the critical step that replaces CHARMM-GUI.**

OpenMM's `addMembrane()` function:
1. Inserts the protein into a POPC lipid bilayer in the XY-plane
2. Adds water molecules above and below the membrane
3. Adds Na+ and Cl- ions to neutralize the system and achieve 0.15 M ionic strength

**Important:** The protein MUST be oriented with the transmembrane domain along the Z-axis. If you downloaded from OPM, it's already oriented. If from RCSB, the cryo-EM structure of 8UO9 is typically already in the correct orientation (transmembrane helices along Z).

⚠️ **This step may take 5-15 minutes depending on your CPU.**

In [ ]:
# Cell 7: Build POPC Membrane + Solvate + Add Ions (FIXED)
from openmm.app import *
from openmm import *
from openmm.unit import *
import time
import numpy as np

print("="*60)
print("  BUILDING MEMBRANE-PROTEIN SYSTEM")
print("="*60)

# Load force field
print("\n[1/5] Loading force field...")
forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')

# Load the protonated protein
print("[2/5] Loading protonated SV2A structure...")
pdb = PDBFile('sv2a_protonated.pdb')
modeller = Modeller(pdb.topology, pdb.positions)

# Calculate the Z-center of the protein to pass to addMembrane
positions_array = np.array([[p.x, p.y, p.z] for p in pdb.positions.value_in_unit(nanometers)])
z_center = float(positions_array[:, 2].mean())
print(f"[3/5] Protein Z-center: {z_center:.3f} nm")

# Build membrane with explicit membraneCenterZ
print("[4/5] Building POPC membrane bilayer...")
print("       (This may take 5-15 minutes, please wait...)")
start_time = time.time()

modeller.addMembrane(
    forcefield,
    lipidType='POPC',
    membraneCenterZ=z_center*nanometers,  # Tell OpenMM where the membrane center is
    minimumPadding=1.0*nanometers,
    ionicStrength=0.15*molar,
    positiveIon='Na+',
    negativeIon='Cl-'
)

elapsed = time.time() - start_time
print(f"       Done! ({elapsed:.1f} seconds)")

# Save the complete system
print("[5/5] Saving assembled system...")
with open('sv2a_membrane_system.pdb', 'w') as f:
    PDBFile.writeFile(modeller.topology, modeller.positions, f)

print("\n" + "="*60)
print("  ✓ SYSTEM BUILT SUCCESSFULLY!")
print("="*60)
print(f"  Output file: sv2a_membrane_system.pdb")
print(f"  Total atoms: {modeller.topology.getNumAtoms()}")


---
## Cell 8: Energy Minimization (GPU)


In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *
import time

print("="*60)
print("  ENERGY MINIMIZATION (GPU)")
print("="*60)

# Load the membrane-protein system
print("\n[1/4] Loading membrane-protein system...")
pdb = PDBFile('sv2a_membrane_system.pdb')
forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')

# Create the System
print("[2/4] Creating system with PME electrostatics...")
system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1.0*nanometers,
    constraints=HBonds
)

# Integrator (not actually used during minimization, but required)
integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.002*picoseconds)

# FORCE CUDA (GPU) PLATFORM
print("[3/4] Setting up CUDA (GPU) platform...")
platform = Platform.getPlatformByName('CUDA')
properties = {'Precision': 'mixed'}   # mixed precision: fast + accurate enough for MD
simulation = Simulation(pdb.topology, system, integrator, platform, properties)
simulation.context.setPositions(pdb.positions)

# Confirm which device is actually being used
print(f"       Using platform: {simulation.context.getPlatform().getName()}")

# Report initial energy
state = simulation.context.getState(getEnergy=True)
initial_energy = state.getPotentialEnergy()
print(f"\n  Initial Potential Energy: {initial_energy}")

# Run minimization
print("\n[4/4] Running Energy Minimization...")
print("       (On GPU this should take well under a minute)")
start_time = time.time()
simulation.minimizeEnergy(maxIterations=5000, tolerance=10*kilojoules_per_mole/nanometer)
elapsed = time.time() - start_time

# Report final energy
state = simulation.context.getState(getEnergy=True)
final_energy = state.getPotentialEnergy()
print(f"\n  Final Potential Energy: {final_energy}")
print(f"  Energy reduction: {initial_energy - final_energy}")
print(f"  Time elapsed: {elapsed:.1f} seconds")

# Save minimized coordinates
positions = simulation.context.getState(getPositions=True).getPositions()
with open('sv2a_minimized.pdb', 'w') as f:
    PDBFile.writeFile(simulation.topology, positions, f)

print("\n  ✓ Minimization complete!")
print("  ✓ Saved: sv2a_minimized.pdb")

---
## Cell 9: NVT Equilibration with Restraints (GPU)

We heat the system from 100 K → 300 K while restraining the protein backbone atoms (C, CA, N) with a harmonic force of 1000 kJ/mol/nm². This allows:
- Lipid tails to pack tightly around the protein
- Water molecules to organize at the membrane-water interface
- The protein structure to remain stable

**Duration:** 50 ps (25,000 steps × 2 fs)  
⚠️ **May take 30-60 minutes on CPU.**

In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *
import time
import sys

print("="*60)
print("  NVT EQUILIBRATION WITH RESTRAINTS (GPU)")
print("="*60)

# Load minimized system
print("\n[1/5] Loading minimized system...")
pdb = PDBFile('sv2a_minimized.pdb')
forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')

# Create System
print("[2/5] Creating system...")
system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1.0*nanometers,
    constraints=HBonds
)

# Add position restraints on protein backbone heavy atoms
print("[3/5] Adding backbone restraints (1000 kJ/mol/nm²)...")
restraint = CustomExternalForce("k*((x-x0)^2 + (y-y0)^2 + (z-z0)^2)")
restraint.addGlobalParameter("k", 1000.0*kilojoules_per_mole/nanometers**2)
restraint.addPerParticleParameter("x0")
restraint.addPerParticleParameter("y0")
restraint.addPerParticleParameter("z0")

# Standard amino acid names
amino_acids = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS',
               'ILE','LEU','LYS','MET','PHE','PRO','SER','THR','TRP',
               'TYR','VAL','HIE','HID','HIP','CYX','ASH','GLH']

n_restrained = 0
for atom in pdb.topology.atoms():
    if atom.residue.name in amino_acids and atom.name in ['C', 'CA', 'N']:
        pos = pdb.positions[atom.index]
        restraint.addParticle(atom.index, [pos[0].value_in_unit(nanometers),
                                           pos[1].value_in_unit(nanometers),
                                           pos[2].value_in_unit(nanometers)])
        n_restrained += 1

system.addForce(restraint)
print(f"       Restrained {n_restrained} backbone atoms")

# Setup integrator starting at 100 K
print("[4/5] Setting up Langevin integrator (start: 100 K)...")
integrator = LangevinMiddleIntegrator(100*kelvin, 1/picosecond, 0.002*picoseconds)

# CUDA (GPU) Platform
platform = Platform.getPlatformByName('CUDA')
properties = {'Precision': 'mixed'}
simulation = Simulation(pdb.topology, system, integrator, platform, properties)
print(f"       Using platform: {simulation.context.getPlatform().getName()}")

simulation.context.setPositions(pdb.positions)
simulation.context.setVelocitiesToTemperature(100*kelvin)

# Heating schedule: 100 K → 300 K in 10 steps
print("\n[5/5] Heating system: 100 K → 300 K...")
print("-" * 40)
start_time = time.time()
steps_per_stage = 2500  # 5 ps per temperature stage
temperatures = list(range(100, 320, 20))  # 100, 120, 140, ... 300
for temp in temperatures:
    integrator.setTemperature(temp*kelvin)
    simulation.step(steps_per_stage)
    state = simulation.context.getState(getEnergy=True)
    pe = state.getPotentialEnergy().value_in_unit(kilojoules_per_mole)
    print(f"  T = {temp:3d} K | PE = {pe:.0f} kJ/mol")
elapsed = time.time() - start_time
print("-" * 40)
print(f"  Time elapsed: {elapsed:.1f} seconds")

# Save NVT equilibrated coordinates
positions = simulation.context.getState(getPositions=True).getPositions()
with open('sv2a_nvt_equil.pdb', 'w') as f:
    PDBFile.writeFile(simulation.topology, positions, f)

print("\n  ✓ NVT Equilibration complete!")
print("  ✓ Saved: sv2a_nvt_equil.pdb")

---
## Cell 10: NPT Equilibration (GPU) — Membrane Relaxation

Now we switch to the **NPT ensemble** to allow the box volume and membrane area to relax. We use a **Monte Carlo Membrane Barostat** which scales XY (membrane plane) isotropically and Z (perpendicular to membrane) independently.

We gradually reduce restraints:
- First 50 ps: Strong restraints (500 kJ/mol/nm²)
- Next 50 ps: Weak restraints (100 kJ/mol/nm²)
- Final 100 ps: No restraints


In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *
import time

print("="*60)
print("  NPT EQUILIBRATION (GPU) — MEMBRANE RELAXATION")
print("="*60)

# Load NVT equilibrated system
print("\n[1/5] Loading NVT-equilibrated system...")
pdb = PDBFile('sv2a_nvt_equil.pdb')
forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')

# Standard amino acid names
amino_acids = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS',
               'ILE','LEU','LYS','MET','PHE','PRO','SER','THR','TRP',
               'TYR','VAL','HIE','HID','HIP','CYX','ASH','GLH']

# --- STAGE 1: Strong restraints (500 kJ/mol/nm²) for 50 ps ---
print("\n[2/5] Stage 1: NPT with strong restraints (500 kJ/mol/nm²) — 50 ps")
system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1.0*nanometers,
    constraints=HBonds
)

# Add membrane barostat (semi-isotropic)
system.addForce(MonteCarloMembraneBarostat(
    1.0*bar,                              # pressure
    0.0*bar*nanometer,                    # surface tension (0 = tensionless membrane)
    300.0*kelvin,                         # temperature
    MonteCarloMembraneBarostat.XYIsotropic,
    MonteCarloMembraneBarostat.ZFree,
    100                                   # frequency
))

# Add restraints
restraint = CustomExternalForce("k*((x-x0)^2 + (y-y0)^2 + (z-z0)^2)")
restraint.addGlobalParameter("k", 500.0*kilojoules_per_mole/nanometers**2)
restraint.addPerParticleParameter("x0")
restraint.addPerParticleParameter("y0")
restraint.addPerParticleParameter("z0")
for atom in pdb.topology.atoms():
    if atom.residue.name in amino_acids and atom.name in ['C', 'CA', 'N']:
        pos = pdb.positions[atom.index]
        restraint.addParticle(atom.index, [pos[0].value_in_unit(nanometers),
                                           pos[1].value_in_unit(nanometers),
                                           pos[2].value_in_unit(nanometers)])
system.addForce(restraint)

integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.002*picoseconds)

# CUDA (GPU) Platform
platform = Platform.getPlatformByName('CUDA')
properties = {'Precision': 'mixed'}
simulation = Simulation(pdb.topology, system, integrator, platform, properties)
print(f"       Using platform: {simulation.context.getPlatform().getName()}")

simulation.context.setPositions(pdb.positions)
simulation.context.setVelocitiesToTemperature(300*kelvin)

start_time = time.time()
simulation.step(25000)  # 50 ps
print(f"       Done. ({time.time()-start_time:.0f} s)")

# Save checkpoint
positions_stage1 = simulation.context.getState(getPositions=True).getPositions()

# --- STAGE 2: Weak restraints (100 kJ/mol/nm²) for 50 ps ---
print("[3/5] Stage 2: NPT with weak restraints (100 kJ/mol/nm²) — 50 ps")
simulation.context.setParameter("k", 100.0*kilojoules_per_mole/nanometers**2)
start_time = time.time()
simulation.step(25000)  # 50 ps
print(f"       Done. ({time.time()-start_time:.0f} s)")

# --- STAGE 3: No restraints for 100 ps ---
print("[4/5] Stage 3: NPT without restraints — 100 ps")
simulation.context.setParameter("k", 0.0*kilojoules_per_mole/nanometers**2)
start_time = time.time()
simulation.step(50000)  # 100 ps
print(f"       Done. ({time.time()-start_time:.0f} s)")

# Save final equilibrated system
print("\n[5/5] Saving final equilibrated system...")
state = simulation.context.getState(getPositions=True, enforcePeriodicBox=True)
positions = state.getPositions()
with open('sv2a_equilibrated.pdb', 'w') as f:
    PDBFile.writeFile(simulation.topology, positions, f)

print("\n" + "="*60)
print("  ✓ NPT EQUILIBRATION COMPLETE!")
print("  ✓ System is GPU-ready: sv2a_equilibrated.pdb")
print("="*60)

# Save full state for the production run (carries positions + velocities + box)
simulation.saveState('sv2a_equilibrated_state.xml')
print("  ✓ Saved continuous state: sv2a_equilibrated_state.xml")

---
## Cell 11: Verify System Stability

Quick check: run 1000 steps and verify no NaN or explosion occurs.

In [ ]:
pdb = PDBFile('sv2a_equilibrated.pdb')
forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
system = forcefield.createSystem(pdb.topology, nonbondedMethod=PME,
                                 nonbondedCutoff=1.0*nanometers, constraints=HBonds)

# tag each force with its own group
for i, f in enumerate(system.getForces()):
    f.setForceGroup(i)
    print(i, type(f).__name__)

integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.002*picoseconds)
sim = Simulation(pdb.topology, system, integrator,
                 Platform.getPlatformByName('CUDA'), {'Precision':'mixed'})
sim.context.setPositions(pdb.positions)

print("\nEnergy by force group:")
for i, f in enumerate(system.getForces()):
    e = sim.context.getState(getEnergy=True, groups={i}).getPotentialEnergy()
    print(f"  {i:2d} {type(f).__name__:30s} {e}")

In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *
import math

pdb = PDBFile('sv2a_equilibrated.pdb')
forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
system = forcefield.createSystem(pdb.topology, nonbondedMethod=PME,
                                 nonbondedCutoff=1.0*nanometers, constraints=HBonds)

integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.002*picoseconds)
sim = Simulation(pdb.topology, system, integrator,
                 Platform.getPlatformByName('CUDA'), {'Precision':'mixed'})
sim.context.setPositions(pdb.positions)

print(f"Before min: {sim.context.getState(getEnergy=True).getPotentialEnergy()}")
sim.minimizeEnergy(maxIterations=5000)
print(f"After min:  {sim.context.getState(getEnergy=True).getPotentialEnergy()}")

# now velocities + short run
sim.context.setVelocitiesToTemperature(300*kelvin)
sim.step(1000)
pe = sim.context.getState(getEnergy=True).getPotentialEnergy()
print("✓ STABLE" if not math.isnan(pe.value_in_unit(kilojoules_per_mole)) else "✗ NaN", pe)

In [ ]:
sim.saveState('sv2a_production_ready.xml')
print("✓ Saved clean state: sv2a_production_ready.xml")

In [ ]:
sim.loadState('sv2a_production_ready.xml')   # exact positions + velocities + box, no overlaps

In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *
import math

pdb = PDBFile('sv2a_equilibrated.pdb')
forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
system = forcefield.createSystem(pdb.topology, nonbondedMethod=PME,
                                 nonbondedCutoff=1.0*nanometers, constraints=HBonds)

integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.002*picoseconds)
platform = Platform.getPlatformByName('CUDA')
properties = {'Precision': 'mixed'}
simulation = Simulation(pdb.topology, system, integrator, platform, properties)
simulation.context.setPositions(pdb.positions)

print(f"Before min: {simulation.context.getState(getEnergy=True).getPotentialEnergy()}")
simulation.minimizeEnergy(maxIterations=5000)          # <-- the line that fixes the overlap
print(f"After min:  {simulation.context.getState(getEnergy=True).getPotentialEnergy()}")

simulation.context.setVelocitiesToTemperature(300*kelvin)
simulation.step(1000)

state = simulation.context.getState(getEnergy=True)
pe = state.getPotentialEnergy()
if math.isnan(pe.value_in_unit(kilojoules_per_mole)):
    print("✗ Still NaN")
else:
    print(f"✓ STABLE. Potential Energy: {pe}")
    simulation.saveState('sv2a_production_ready.xml')   # save the clean state right away
    print("✓ Saved: sv2a_production_ready.xml")

In [ ]:
pip install openmmforcefields openff-toolkit

---
## Cell 12: Production MD Script (GPU)

**Copy this script to your GPU workstation/cluster** and run it there.

This runs a 100 ns production simulation using the CUDA platform.

In [ ]:
# Save this as a standalone script for your GPU machine

gpu_script = '''
#!/usr/bin/env python
"""
SV2A Production MD Simulation (GPU)
Run this on your GPU workstation/cluster.
Requires: sv2a_equilibrated.pdb in the same directory.
"""
from openmm.app import *
from openmm import *
from openmm.unit import *
import sys

# ============================================
# CONFIGURATION
# ============================================
INPUT_PDB = "sv2a_equilibrated.pdb"
OUTPUT_DCD = "sv2a_production.dcd"
OUTPUT_LOG = "sv2a_production.log"
CHECKPOINT = "sv2a_checkpoint.chk"

TIMESTEP = 0.002  # ps (2 fs)
TOTAL_TIME_NS = 100  # nanoseconds
SAVE_FREQ_PS = 10  # save frame every 10 ps
LOG_FREQ_PS = 10   # log energy every 10 ps
CHECKPOINT_FREQ_PS = 100  # checkpoint every 100 ps

# Calculate steps
total_steps = int(TOTAL_TIME_NS * 1000 / TIMESTEP)  # 50,000,000 steps
save_freq = int(SAVE_FREQ_PS / TIMESTEP)  # 5000 steps
log_freq = int(LOG_FREQ_PS / TIMESTEP)    # 5000 steps
chk_freq = int(CHECKPOINT_FREQ_PS / TIMESTEP)  # 50000 steps

# ============================================
# SETUP
# ============================================
print("Loading system...")
pdb = PDBFile(INPUT_PDB)
forcefield = ForceField("amber14-all.xml", "amber14/tip3pfb.xml")

system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1.0*nanometers,
    constraints=HBonds
)

# Semi-isotropic barostat for membrane
system.addForce(MonteCarloMembraneBarostat(
    1.0*bar, 300.0*kelvin,
    MonteCarloMembraneBarostat.XYIsotropic,
    MonteCarloMembraneBarostat.ZFree,
    100
))

integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, TIMESTEP*picoseconds)

# Try CUDA first, fall back to OpenCL
try:
    platform = Platform.getPlatformByName("CUDA")
    properties = {"CudaPrecision": "mixed"}
    print("Using CUDA platform (GPU)")
except:
    try:
        platform = Platform.getPlatformByName("OpenCL")
        properties = {}
        print("Using OpenCL platform (GPU)")
    except:
        platform = Platform.getPlatformByName("CPU")
        properties = {}
        print("WARNING: No GPU found, using CPU (will be very slow!)")

simulation = Simulation(pdb.topology, system, integrator, platform, properties)
simulation.context.setPositions(pdb.positions)
simulation.context.setVelocitiesToTemperature(300*kelvin)

# ============================================
# REPORTERS
# ============================================
simulation.reporters.append(DCDReporter(OUTPUT_DCD, save_freq))
simulation.reporters.append(CheckpointReporter(CHECKPOINT, chk_freq))
simulation.reporters.append(StateDataReporter(
    OUTPUT_LOG, log_freq,
    step=True, time=True, potentialEnergy=True, kineticEnergy=True,
    temperature=True, volume=True, density=True, speed=True
))
simulation.reporters.append(StateDataReporter(
    sys.stdout, log_freq*10,
    step=True, time=True, temperature=True, speed=True, remainingTime=True,
    totalSteps=total_steps
))

# ============================================
# RUN PRODUCTION
# ============================================
print(f"\\nStarting {TOTAL_TIME_NS} ns production run ({total_steps:,} steps)...")
print(f"Trajectory: {OUTPUT_DCD} (frame every {SAVE_FREQ_PS} ps)")
print(f"Log: {OUTPUT_LOG}")
print(f"Checkpoint: {CHECKPOINT}\\n")

simulation.step(total_steps)

print("\\n" + "="*50)
print("  PRODUCTION SIMULATION COMPLETE!")
print("="*50)
'''

with open('run_production_gpu.py', 'w') as f:
    f.write(gpu_script)

print("✓ GPU production script saved: run_production_gpu.py")
print("\nTo run on your GPU machine:")
print("  1. Copy 'sv2a_equilibrated.pdb' and 'run_production_gpu.py' to GPU")
print("  2. Activate your conda environment with OpenMM")
print("  3. Run: python run_production_gpu.py")

---
## Summary of Files Generated

| Step | File | Description |
|------|------|-------------|
| Download | `8UO9.pdb` | Raw PDB from RCSB |
| Clean | `sv2a_clean.pdb` | Chain A only, formatted |
| Protonate | `sv2a_protonated.pdb` | Hydrogens added at pH 7 |
| Membrane | `sv2a_membrane_system.pdb` | Protein + POPC + water + ions |
| Minimize | `sv2a_minimized.pdb` | Energy minimized |
| NVT | `sv2a_nvt_equil.pdb` | Heated to 300 K |
| NPT | `sv2a_equilibrated.pdb` | **GPU-READY** system |
| Script | `run_production_gpu.py` | Production script for GPU |

---

## Troubleshooting

| Error | Solution |
|-------|----------|
| `AttributeError: NoneType _finalize` | PDB file is malformed. Re-run Cell 4 (cleaning step). |
| `Energy is NaN` | Increase minimization steps to 10000. |
| `addMembrane() fails` | Protein may not be oriented along Z. Use OPM structure. |
| `Out of memory` on CPU | Reduce padding to 0.8 nm in addMembrane(). |
| `Force field missing residue` | Non-standard residues present. Set `keep_ligands=False` in Cell 4. |